In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib


In [7]:
matches = pd.read_csv("../data/matches.csv")
deliveries = pd.read_csv("../data/deliveries.csv")


In [8]:
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

matches.drop_duplicates(inplace=True)
deliveries.drop_duplicates(inplace=True)


In [9]:
batting_stats = deliveries.groupby(
    ['match_id', 'batter']
).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count'),
    fours=('batsman_runs', lambda x: (x == 4).sum()),
    sixes=('batsman_runs', lambda x: (x == 6).sum())
).reset_index()


In [10]:
bowling_stats = deliveries.groupby(
    ['match_id', 'bowler']
).agg(
    runs_conceded=('total_runs', 'sum'),
    balls_bowled=('ball', 'count'),
    wickets=('is_wicket', 'sum')
).reset_index()


In [11]:
player_match = pd.merge(
    batting_stats,
    bowling_stats,
    left_on=['match_id', 'batter'],
    right_on=['match_id', 'bowler'],
    how='outer'
)

player_match.rename(columns={'batter': 'player'}, inplace=True)
player_match.drop(columns=['bowler'], inplace=True)
player_match.fillna(0, inplace=True)


In [12]:
player_match = player_match.merge(
    matches[['id', 'date', 'venue', 'team1', 'team2']],
    left_on='match_id',
    right_on='id',
    how='left'
)

player_match.drop(columns='id', inplace=True)
player_match.sort_values(['player', 'date'], inplace=True)


In [13]:
player_match['rolling_runs_5'] = (
    player_match
    .groupby('player')['runs_scored']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)


In [14]:
venue_avg = player_match.groupby(
    ['player', 'venue']
)['runs_scored'].mean().reset_index(name='venue_avg_runs')

player_match = player_match.merge(
    venue_avg,
    on=['player', 'venue'],
    how='left'
)


In [15]:
player_match['opponent'] = np.where(
    player_match['team1'] == player_match['player'],
    player_match['team2'],
    player_match['team1']
)

opponent_avg = player_match.groupby(
    ['player', 'opponent']
)['runs_scored'].mean().reset_index(name='opp_avg_runs')

player_match = player_match.merge(
    opponent_avg,
    on=['player', 'opponent'],
    how='left'
)


In [16]:
player_match['career_runs_avg'] = (
    player_match
    .groupby('player')['runs_scored']
    .transform('mean')
)


In [17]:
player_match['target_next_runs'] = (
    player_match
    .groupby('player')['runs_scored']
    .shift(-1)
)


In [18]:
player_match.dropna(subset=['target_next_runs'], inplace=True)


In [19]:
train = player_match[player_match['date'] < '2019-01-01']
test = player_match[player_match['date'] >= '2019-01-01']


In [20]:
num_features = [
    'rolling_runs_5',
    'venue_avg_runs',
    'opp_avg_runs',
    'career_runs_avg'
]


In [21]:
numeric_transformer = Pipeline(
    steps=[('scaler', StandardScaler())]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features)
    ]
)
